In [ ]:
# setup
import urllib.request
import pandas as pd
import numpy as np

In [ ]:
# Get the data
meshblocks_url = "https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021/Mesh%20Block%20Counts%2C%202021.xlsx"
file = "ABS 2021 Mesh Block Counts.xlsx"
urllib.request.urlretrieve(meshblocks_url, file)

In [ ]:
sheets = {
"New South Wales"               :[{"sheet_name":"Table 1","header":7,"skip_footer":4},{"sheet_name":"Table 1.1","header":7,"skip_footer":4}],
"Victoria"                      :[{"sheet_name":"Table 2","header":7,"skip_footer":4},{"sheet_name":"Table 2.1","header":7,"skip_footer":4}],
"Queensland"                    :[{"sheet_name":"Table 3","header":7,"skip_footer":4},{"sheet_name":"Table 3.1","header":7,"skip_footer":4}],
"South Australia"               :[{"sheet_name":"Table 4","header":7,"skip_footer":4}],
"Western Australia"             :[{"sheet_name":"Table 5","header":7,"skip_footer":4}],
"Tasmania"                      :[{"sheet_name":"Table 6","header":7,"skip_footer":4}],
"Northern Territory"            :[{"sheet_name":"Table 7","header":7,"skip_footer":4}],
"Australian Capital Territory"  :[{"sheet_name":"Table 8","header":7,"skip_footer":4}],
"Other Territories"             :[{"sheet_name":"Table 9","header":7,"skip_footer":4}],
}
column_types = {
'MB_CODE_2021'         :'object',
'MB_CATEGORY_NAME_2021':'object',
'AREA_ALBERS_SQKM'     :'float64',
'Dwelling'             :'Int64',
'Person'               :'Int64',
'State'                :'Int64',
}

In [ ]:
# Load up Mesh Block count dataframes for each State and Territory
dfs = {}
for state in sheets.keys():
    for s in sheets[state]:
        df = pd.read_excel(file, 
                           sheet_name=s['sheet_name'],
                           header=s['header']-1,
                           usecols="A:F",
                           dtype=column_types,
                           skipfooter=s['skip_footer'])
        if state not in dfs.keys():
            dfs[state]=df.copy()
        else:
            dfs[state]=dfs[state].append(df.copy())
# Combine Mesh Block count dataframes
df = pd.concat(dfs)
# drop the within-state sequential index
df = df.droplevel(1).reset_index()
# Rename the State/Territory identifier columns
df.columns = ['STATE_NAME_2021'] + list(column_types.keys())[:-1] + ['STATE_CODE_2021']
# Re-order the columns so State name comes after Mesh Block Category
# State name will be retained so people don't have to mess around looking up State codes
df = df[['MB_CODE_2021',
 'MB_CATEGORY_NAME_2021',
 'STATE_CODE_2021',
 'STATE_NAME_2021',
 'AREA_ALBERS_SQKM',
 'Dwelling',
 'Person']]
df

In [ ]:
# output national combined Mesh Block counts dataset
df.to_csv(f"{file.split('.')[0]}.csv",index=False)
# output state specific Mesh Block counts datasets
for state in dfs.keys():
    dfs[state].to_csv(f"{file.split('.')[0]} - {state}.csv",index=False)
df.groupby('STATE_NAME_2021')[['AREA_ALBERS_SQKM','Dwelling','Person']].sum()